# SetFit — CPU-efficient training

Same data, split, and evaluation as `setfit_training.ipynb`, tuned to train on
CPU: a short token window, few contrastive pairs, and a scikit-learn head (no
torch head to train). `pip install setfit`

Model: **`google/embeddinggemma-300m`** — top MTEB model under 500M params
(Sep 2025), designed for on-device/CPU. It's gated and ~14× larger than MiniLM,
so if training is too slow, drop to `BAAI/bge-small-en-v1.5` (33M) — both are
one-line swaps in the training cell.

In [ ]:
import json
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

df = pd.read_csv('labeled_demo.csv')
# CSV stores "category::label" -> keep just the label name
df['labels'] = df['labels'].apply(lambda s: [x.split('::')[-1] for x in json.loads(s)])

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['labels'])              # rows -> multi-hot label matrix
assert Y.shape == (len(df), len(mlb.classes_))
print(len(df), 'rows,', list(mlb.classes_))

In [ ]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

# EmbeddingGemma (Sep 2025) — #1 on MTEB under 500M, built for on-device/CPU, Matryoshka dims.
# Gated: run `huggingface-cli login` and accept the license at hf.co/google/embeddinggemma-300m
MODEL_NAME = 'google/embeddinggemma-300m'              # 308M, best quality
# MODEL_NAME = 'BAAI/bge-small-en-v1.5'                # 33M, ungated, fastest recent small option
# MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # 22M, lightest fallback

# same split + seed as the other notebooks -> identical test set
text_train, text_test, Y_train, Y_test = train_test_split(
    df['text'].tolist(), Y, test_size=0.3, random_state=42)
train_ds = Dataset.from_dict({'text': text_train, 'label': Y_train.tolist()})   # multi-hot

model = SetFitModel.from_pretrained(MODEL_NAME, multi_target_strategy='one-vs-rest',
                                    use_differentiable_head=False)  # sklearn head, no torch head training
model.model_body.max_seq_length = 128            # cap tokens -> less compute per text
# model.model_body.truncate_dim = 256            # EmbeddingGemma only: 768->256 dims, faster, ~same quality

args = TrainingArguments(
    batch_size=(8, 16),     # (embedding, head) batch; small embedding batch saves RAM
    num_iterations=5,       # few contrastive pairs per example -> fast (overrides sampling_strategy)
    num_epochs=1,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

pred = np.asarray(model.predict(text_test))
print(classification_report(Y_test, pred, target_names=mlb.classes_, zero_division=0))
print('exact-match row accuracy:', round(accuracy_score(Y_test, pred), 3))

In [ ]:
results = pd.DataFrame({
    'text': text_test,
    'labels': [sorted(l) for l in mlb.inverse_transform(Y_test)],
    'predicted': [sorted(p) for p in mlb.inverse_transform(pred)],
})
results['is_correct'] = [set(t) == set(p)
                         for t, p in zip(results['labels'], results['predicted'])]
results